Laboratorio 8:

1. Definimos el esquema 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Region_Type", StringType(), True),
    StructField("Family_Size", IntegerType(), True),
    StructField("Parent_Education", StringType(), True),
    StructField("Family_Income_Level", StringType(), True),
    StructField("Internet_Quality", DoubleType(), True),
    StructField("Study_Space_Quality", DoubleType(), True),
    StructField("Previous_GPA", DoubleType(), True),
    StructField("Number_of_Failed_Courses", IntegerType(), True),
    StructField("Total_Credits_Earned", IntegerType(), True),
    StructField("Weekly_Study_Hours", DoubleType(), True),
    StructField("Attendance_Rate", DoubleType(), True),
    StructField("Library_Visits_Per_Month", IntegerType(), True),
    StructField("Extracurricular_Hours", DoubleType(), True),
    StructField("Sleep_Hours", DoubleType(), True),
    StructField("Social_Media_Usage_Hours", DoubleType(), True),
    StructField("Stress_Level", DoubleType(), True),
    StructField("Motivation_Score", DoubleType(), True),
    StructField("Self_Efficacy_Score", DoubleType(), True),
    StructField("Midterm_Mark", DoubleType(), True),
    StructField("Final_Exam_Score", DoubleType(), True),
])


In [0]:
from pyspark.sql.functions import col

# Leer CSV con header, sin forzar schema (permite inferencia)
df_estudiantes = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion_07/csv")
    .withColumn("archivo_origen", col("_metadata.file_path"))
)

2 Escrbir a formato parquet

In [0]:
df_estudiantes.write.format("parquet").mode("overwrite").save("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion_07/parquet")

3. Leer formato parquet 

In [0]:
df_estudiantes_parquet = spark.read.format("parquet").load("/Volumes/proyecto_estudiantes/bronze/volumen_estudiantes/sesion_07/parquet")

df_estudiantes_parquet.count()

4. Escribir formaro delta con partición

In [0]:
df_estudiantes_parquet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("Region_Type").saveAsTable("proyecto_estudiantes.default.estudiantes_con_particion")

5. Escribir formato delta sin partición

In [0]:
df_estudiantes_parquet.write.format("delta").mode("overwrite").saveAsTable("tecsup.default.estudiantes_sin_particion")

In [0]:
%sql
SELECT Region_Type, COUNT(1) AS cantidad_estudiantes
FROM tecsup.default.estudiantes_sin_particion
GROUP BY Region_Type

In [0]:
%sql
select *
from tecsup.default.estudiantes_sin_particion
WHERE Region_Type = 'Rural'

In [0]:
%sql
select *
from proyecto_estudiantes.default.estudiantes_con_particion
WHERE Region_Type = 'Rural'

6. Time Travel

In [0]:
%sql
SELECT Family_Income_Level, COUNT(1) AS cantidad_estudiantes
FROM proyecto_estudiantes.default.estudiantes_con_particion
WHERE Gender = 'Female'
GROUP BY Family_Income_Level

In [0]:
%sql
UPDATE proyecto_estudiantes.default.estudiantes_con_particion
SET Family_Income_Level = 'High'
WHERE Previous_GPA > 3.0

In [0]:
%sql
SELECT Gender, COUNT(1) AS cantidad_estudiantes
FROM proyecto_estudiantes.default.estudiantes_con_particion
WHERE Region_Type = 'Urban'
GROUP BY Gender

In [0]:
%sql
DESCRIBE HISTORY proyecto_estudiantes.default.estudiantes_con_particion

In [0]:
%sql
RESTORE TABLE proyecto_estudiantes.default.estudiantes_con_particion VERSION AS OF 0

In [0]:
%sql
DESCRIBE HISTORY proyecto_estudiantes.default.estudiantes_con_particion

In [0]:
%sql
SELECT Gender, COUNT(1) AS cantidad_estudiantes
FROM proyecto_estudiantes.default.estudiantes_con_particion
WHERE Region_Type = 'Urban'
GROUP BY Gender